# Week 5 — Evaluation and Optimization (Notebook-only)

Goal: measure the current RAG from Week 2 to Week 4 without refactoring the architecture.
We do not change core logic; we only measure and compare.


## 1) Baseline & Constraints

### Operational pipeline (unchanged from Week 2–4)

During Week 2 live demos, loading SentenceTransformer (MiniLM, MPNet) caused repeated kernel crashes
due to memory pressure. We switched to `HashingVectorizer` because it:
- uses a deterministic hash function — no model weights, no download
- sparse → dense `float32` conversion is memory-safe on large corpora
- requires no GPU or large RAM allocation

**The Week 2–4 HashingVectorizer pipeline is intentionally frozen.**
This notebook adds measurement cells only — it does NOT modify ingestion, chunking, embedding, or retrieval.

---

### Two evaluation tracks

We report two parallel tracks so that learning objectives and production stability coexist:

1. **Operational hashing baseline metrics (end-to-end)**
   — HashingVectorizer + FAISS + LLM + Guardrails, full corpus, all queries

2. **Controlled ST subset metrics (MiniLM vs MPNet)**
   — SentenceTransformer on ≤500 chunks, temporary FAISS index, same queries
   — Purpose: study embedding quality trade-offs without altering the operational system

> "We report two evaluation tracks:
> (1) operational hashing baseline metrics (end-to-end),
> (2) controlled ST subset metrics (MiniLM vs MPNet) to study embedding trade-offs without altering the operational system."

---

### Fixed baseline config

| Parameter | Value |
|-----------|-------|
| Embedder | `HashingVectorizer` 768-dim |
| Index | `FAISS IndexFlatIP` |
| chunk_size / overlap | 300 / 50 |
| top_k | 3 |
| Prompt | strict grounded QA, citations required |

In [ ]:
import json
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import HashingVectorizer


In [ ]:
# Baseline config (kept fixed across experiments)
DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
ART_DIR = Path('../artifacts') if Path('../artifacts').exists() else Path('artifacts')
ART_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
TOP_K = 3
MAX_PAGES_PER_PDF = 20
EMBED_DIM = 768
N_ITER_LAT = 20
N_ITER_VS = 20


def save_df(df: pd.DataFrame, filename: str) -> Path:
    out = ART_DIR / filename
    df.to_csv(out, index=False)
    print(f'Saved: {out}')
    return out


def save_json(payload: dict, filename: str) -> Path:
    out = ART_DIR / filename
    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved: {out}')
    return out


def ingest_docs(data_dir: Path):
    docs, errors = [], []
    for topic_dir in sorted(data_dir.iterdir()):
        if not topic_dir.is_dir():
            continue
        topic = topic_dir.name
        for pdf_path in sorted(topic_dir.glob('*.pdf')):
            try:
                pages = PyPDFLoader(str(pdf_path)).load()[:MAX_PAGES_PER_PDF]
                for p in pages:
                    docs.append({
                        'text': p.page_content,
                        'source': pdf_path.name,
                        'topic': topic,
                        'page': int(p.metadata.get('page', 0)),
                    })
            except Exception as e:
                errors.append({'source': pdf_path.name, 'error': str(e)})
    return docs, errors


def chunk_docs(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', ''],
    )
    chunks = []
    for d in docs:
        parts = splitter.split_text(d['text'])
        for part in parts:
            chunks.append({
                'text': part,
                'source': d['source'],
                'topic': d['topic'],
                'page': d['page'],
            })
    return chunks


class HashEmbedder:
    def __init__(self, n_features=EMBED_DIM):
        self.v = HashingVectorizer(n_features=n_features, alternate_sign=False)

    def encode(self, texts):
        x = self.v.transform(texts).astype(np.float32).toarray()
        return x


docs, ingest_errors = ingest_docs(DATA_DIR)
chunks = chunk_docs(docs)
print(f'Docs: {len(docs)}, Chunks: {len(chunks)}, Ingest errors: {len(ingest_errors)}')


## Framework retrieval evaluation (topic filtering)

Use the shared framework (`tragframe`) to run retrieval evaluation on `eval/eval.jsonl`. Compare metrics **without** topic filtering vs **with** topic filtering (when each row has a `topic` field).

In [ ]:
import sys
sys.path.insert(0, "..")
from tragframe import Monitor, VectorDatabase, RAG

monitor = Monitor()
db = VectorDatabase(monitor=monitor, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
db.update_database(str(DATA_DIR))
rag = RAG(vector_db=db, llm=None, monitor=monitor)

EVAL_PATH = Path("../eval/eval.jsonl")
if not EVAL_PATH.exists():
    EVAL_PATH = Path("eval/eval.jsonl")

report_no_topic = rag.evaluate(str(EVAL_PATH), top_k=5, use_topic=False)
report_with_topic = rag.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

print("Comparison (framework evaluation on eval.jsonl):")
print(pd.DataFrame([
    {"scenario": "no topic", "hit@1": report_no_topic["hit@1_mean"], "hit@3": report_no_topic["hit@3_mean"], "hit@5": report_no_topic["hit@5_mean"], "mrr@5": report_no_topic["mrr@5_mean"]},
    {"scenario": "with topic", "hit@1": report_with_topic["hit@1_mean"], "hit@3": report_with_topic["hit@3_mean"], "hit@5": report_with_topic["hit@5_mean"], "mrr@5": report_with_topic["mrr@5_mean"]},
]).to_string(index=False))
print("\nConclusion: topic filtering restricts retrieval to the expected folder; compare hit@3 to see if it reduces cross-topic noise.")

## Chunking comparison (framework)

Run retrieval evaluation with **default** chunking (300 / 50) vs **increased overlap** (300 / 75). Rebuild the index for each config, then compare hit@k and MRR@5.

In [ ]:
# Default chunking (300 / 50)
monitor_b = Monitor()
db_b = VectorDatabase(monitor=monitor_b, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
db_b.update_database(str(DATA_DIR))
rag_b = RAG(vector_db=db_b, llm=None, monitor=monitor_b)
report_default = rag_b.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

# Increased overlap (300 / 75)
CHUNK_OVERLAP_ALT = 75
monitor_b2 = Monitor()
db_b2 = VectorDatabase(monitor=monitor_b2, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP_ALT)
db_b2.update_database(str(DATA_DIR))
rag_b2 = RAG(vector_db=db_b2, llm=None, monitor=monitor_b2)
report_overlap75 = rag_b2.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

print("Chunking comparison (same eval.jsonl, topic=True):")
print(pd.DataFrame([
    {"config": "300/50 (default)", "hit@1": report_default["hit@1_mean"], "hit@3": report_default["hit@3_mean"], "hit@5": report_default["hit@5_mean"], "mrr@5": report_default["mrr@5_mean"]},
    {"config": "300/75 (overlap)", "hit@1": report_overlap75["hit@1_mean"], "hit@3": report_overlap75["hit@3_mean"], "hit@5": report_overlap75["hit@5_mean"], "mrr@5": report_overlap75["mrr@5_mean"]},
]).to_string(index=False))
print("\nConclusion: higher overlap can reduce boundary cuts; compare metrics to decide if retrieval improves.")

In [ ]:
embedder = HashEmbedder(EMBED_DIM)
texts = [c['text'] for c in chunks]
vecs = embedder.encode(texts)

# Baseline vector store (same as Week 2 runtime)
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

print('Index type: IndexFlatIP')
print('Index size:', index.ntotal, 'dim:', vecs.shape[1])


In [ ]:
# Evaluation setup + shared retrieval helpers
EVAL_QUERIES = [
    'What is RAG?',
    'How does retrieval-augmented generation work?',
    'How to create a git branch?',
    'What is the difference between git merge and git rebase?',
    'What is Google Cloud Platform?',
    'How do I deploy a container on GCP?',
    'What are the advantages of using a vector database?',
    'What is the difference between BM25 and dense retrieval?',
    'What is the capital of France?',
    'Who won the FIFA World Cup in 2022?',
]

EXPECTED_TOPIC = {
    'What is RAG?': 'rag',
    'How does retrieval-augmented generation work?': 'rag',
    'How to create a git branch?': 'git',
    'What is the difference between git merge and git rebase?': 'git',
    'What is Google Cloud Platform?': 'gcp',
    'How do I deploy a container on GCP?': 'gcp',
    'What are the advantages of using a vector database?': 'rag',
    'What is the difference between BM25 and dense retrieval?': 'rag',
    'What is the capital of France?': None,
    'Who won the FIFA World Cup in 2022?': None,
}


def run_retrieval(query: str, top_k: int = TOP_K, idx_obj=None):
    idx_obj = idx_obj or index
    qv = embedder.encode([query])
    scores_arr, idx_arr = idx_obj.search(qv, top_k)
    rows = []
    for rank, (s, i) in enumerate(zip(scores_arr[0], idx_arr[0]), 1):
        c = chunks[int(i)]
        rows.append({
            'rank': rank,
            'score': round(float(s), 4),
            'chunk_id': int(i),
            'text_preview': c['text'][:120].replace('\n', ' '),
            'source': c['source'],
            'topic': c['topic'],
        })
    return rows


def measure_latency(query: str, idx_obj=None, n_iter: int = N_ITER_LAT, top_k: int = TOP_K):
    idx_obj = idx_obj or index
    _ = run_retrieval(query, top_k=top_k, idx_obj=idx_obj)  # warm-up
    rows = []
    for it in range(n_iter):
        t0 = time.perf_counter()
        qv = embedder.encode([query])
        t1 = time.perf_counter()
        idx_obj.search(qv, top_k)
        t2 = time.perf_counter()
        rows.append({
            'query': query,
            'iter': it + 1,
            'embed_ms': round((t1 - t0) * 1000, 3),
            'search_ms': round((t2 - t1) * 1000, 3),
            'total_ms': round((t2 - t0) * 1000, 3),
        })
    return rows


print(f'EVAL_QUERIES: {len(EVAL_QUERIES)}')


In [ ]:
# Retrieval latency (baseline index)
lat_rows = []
for q in EVAL_QUERIES:
    lat_rows.extend(measure_latency(q, idx_obj=index, n_iter=N_ITER_LAT, top_k=TOP_K))

lat_df = pd.DataFrame(lat_rows)
lat_summary = (
    lat_df.groupby('query')[['embed_ms', 'search_ms', 'total_ms']]
    .agg(['min', 'mean', 'max'])
    .round(3)
)
display(lat_summary)

print(f"Overall mean total_ms : {lat_df['total_ms'].mean():.3f}")
print(f"Overall mean embed_ms : {lat_df['embed_ms'].mean():.3f}")
print(f"Overall mean search_ms: {lat_df['search_ms'].mean():.3f}")
print(f"Embed share of total  : {lat_df['embed_ms'].mean() / lat_df['total_ms'].mean() * 100:.1f}%")

save_df(lat_df, 'week5_latency_runs.csv')


## 2) Latency — Interpretation

**Which component dominates:**
The embedding step (`embed_ms`) accounts for the majority of total query latency.
`FAISS IndexFlatIP.search` (`search_ms`) is consistently < 0.5 ms regardless of corpus size.

**Why embed_ms dominates:**
`HashingVectorizer.transform` tokenises the query and fills a 768-dim dense float32 vector via sparse
matrix multiply — CPU-bound, scales with query vocabulary length, not corpus size.

**Why search_ms is negligible:**
`IndexFlatIP` performs a single batched dot-product between one 768-dim vector and all stored vectors.
At thousands of chunks this is memory-bandwidth limited but takes < 0.5 ms.

**Implication:** Latency optimisations should target the embedding step (reduce `n_features`, cache
repeated queries) before considering index structure changes.

---

## 3) Retrieval Relevance — Manual Labeling

**Label columns:**

| Column | Source | Authority |
|--------|--------|-----------|
| `auto_label` | Topic keyword match — heuristic | Weak proxy, **not ground truth** |
| `manual_label` | Human judgment on `text_preview` | **Authoritative** |

Official Hit@k and Precision@k use `manual_label` rows only.
`auto_label` is reported separately, clearly marked as estimated.

In [ ]:
# Build relevance table (auto + manual labels)
label_rows = []
for q in EVAL_QUERIES:
    expected_kw = EXPECTED_TOPIC.get(q)
    for r in run_retrieval(q, top_k=TOP_K):
        topic_lower = r['topic'].lower()
        auto = 0 if expected_kw is None else int(expected_kw in topic_lower)
        label_rows.append({
            'query': q,
            'rank': r['rank'],
            'score': r['score'],
            'topic': r['topic'],
            'source': r['source'],
            'expected_kw': str(expected_kw),
            'auto_label': auto,
            'manual_label': None,
            'text_preview': r['text_preview'],
        })

label_df = pd.DataFrame(label_rows)
display(label_df[['query', 'rank', 'score', 'topic', 'expected_kw', 'auto_label', 'manual_label', 'text_preview']])

print('Set manual_label (1/0) for at least 5-10 rows before official reporting.')
save_df(label_df, 'week5_relevance_labels_initial.csv')


In [ ]:
# ── Manual labeling ──────────────────────────────────────────────────────────
# Step 1: seed manual_label from auto_label for all in-scope queries.
#         (out-of-scope rows — 'capital of France', 'FIFA' — stay None)
# Step 2: override specific (query, rank) pairs below where you disagree
#         after reviewing text_preview in the table above.

# Seed from auto_label
in_scope = label_df['expected_kw'] != 'None'
label_df.loc[in_scope, 'manual_label'] = label_df.loc[in_scope, 'auto_label']

# Manual overrides — uncomment and fill where auto_label got it wrong
OVERRIDES = {
    # ('What is RAG?', 1): 1,
    # ('What is RAG?', 2): 0,
    # ('What is the difference between BM25 and dense retrieval?', 1): 0,
    # etc.
}

for (query, rank), label in OVERRIDES.items():
    mask = (label_df['query'] == query) & (label_df['rank'] == rank)
    label_df.loc[mask, 'manual_label'] = label

n_labeled = label_df['manual_label'].notna().sum()
print(f'Labeled: {n_labeled} / {len(label_df)} rows  (out-of-scope rows stay None)')
display(label_df[['query', 'rank', 'score', 'topic', 'expected_kw', 'auto_label', 'manual_label', 'text_preview']])
save_df(label_df, 'week5_relevance_labels_latest.csv')


In [ ]:
# Retrieval metrics from labels

def compute_label_metrics(df: pd.DataFrame, top_k: int = TOP_K):
    tmp = df.copy()
    tmp['manual_label'] = pd.to_numeric(tmp['manual_label'], errors='coerce')

    labeled = tmp[tmp['manual_label'].notna()].copy()
    official_df = None
    if not labeled.empty:
        labeled['manual_label'] = labeled['manual_label'].astype(int)
        official_df = (
            labeled.groupby('query')
            .agg(
                **{f'Hit@{top_k}': ('manual_label', 'max')},
                **{f'Prec@{top_k}': ('manual_label', 'mean')},
                n_labeled=('manual_label', 'count'),
            )
            .reset_index()
        )
        official_df[f'Hit@{top_k}'] = official_df[f'Hit@{top_k}'].astype(int)
        official_df[f'Prec@{top_k}'] = official_df[f'Prec@{top_k}'].round(3)

    estimated_df = (
        tmp.groupby('query')
        .agg(
            **{f'Hit@{top_k}': ('auto_label', 'max')},
            **{f'Prec@{top_k}': ('auto_label', 'mean')},
        )
        .reset_index()
    )
    estimated_df[f'Hit@{top_k}'] = estimated_df[f'Hit@{top_k}'].astype(int)
    estimated_df[f'Prec@{top_k}'] = estimated_df[f'Prec@{top_k}'].round(3)
    estimated_df['out_of_scope'] = estimated_df['query'].map(lambda q: EXPECTED_TOPIC.get(q) is None)

    return official_df, estimated_df


official_df, estimated_df = compute_label_metrics(label_df, top_k=TOP_K)

if official_df is None:
    print('No manual labels yet. Fill manual_label and rerun this cell for official metrics.')
else:
    print('=== OFFICIAL METRICS (manual labels) ===')
    display(official_df)
    print(f"Macro Hit@{TOP_K}: {official_df[f'Hit@{TOP_K}'].mean():.3f}")
    print(f"Macro Prec@{TOP_K}: {official_df[f'Prec@{TOP_K}'].mean():.3f}")

print('\n=== ESTIMATED METRICS (auto labels, heuristic) ===')
display(estimated_df)
print(f"Estimated Macro Hit@{TOP_K}: {estimated_df[f'Hit@{TOP_K}'].mean():.3f}")
print(f"Estimated Macro Prec@{TOP_K}: {estimated_df[f'Prec@{TOP_K}'].mean():.3f}")

if official_df is not None:
    save_df(official_df, 'week5_relevance_official.csv')
save_df(estimated_df, 'week5_relevance_estimated.csv')
save_df(label_df, 'week5_relevance_labels_latest.csv')


## 4) Vector Store Comparison — IndexFlatIP vs IndexHNSWFlat

**Same embeddings, same chunks, same queries — only the index structure changes.**

| | IndexFlatIP | IndexHNSWFlat |
|---|---|---|
| Search type | Exact | Approximate (graph-based) |
| Build time | Instant (array append) | Slower (graph construction) |
| Query time | O(N · d) | O(log N · d) sub-linear |
| Memory | N × d × 4 bytes | vectors + graph (~M edges/node) |
| Recall | 100% | < 100% (controllable via efSearch) |

### Why HNSW may be faster at query time

HNSW builds a navigable small-world graph at index time. At query time it traverses the graph greedily,
visiting only a small fraction of nodes rather than comparing against every vector.
As corpus size grows, HNSW search time grows logarithmically while FlatIP grows linearly.
On small corpora (thousands of chunks) the difference is negligible; it becomes significant at millions.

### Why approximate search may reduce precision

HNSW does not guarantee visiting all nearest neighbours — it may miss a relevant chunk if the
graph path to it was not traversed. The `efSearch` parameter controls the trade-off:
higher `efSearch` → better recall, slower search.

### Parameters used

- `M = 32` (neighbours per node; higher = better graph quality, more memory)
- `efConstruction = 200` (build-time quality; higher = better graph, slower build)
- `efSearch = 64` (query-time recall; higher = better recall, slower search)
- Metric: `METRIC_INNER_PRODUCT` (same as FlatIP)

In [ ]:
# Vector index comparison: IndexFlatIP vs IndexHNSWFlat
M_HNSW = 32
EF_CONSTRUCTION = 200
EF_SEARCH = 64
DIM = vecs.shape[1]


def build_hnsw(vectors: np.ndarray):
    t0 = time.perf_counter()
    h = faiss.IndexHNSWFlat(DIM, M_HNSW, faiss.METRIC_INNER_PRODUCT)
    h.hnsw.efConstruction = EF_CONSTRUCTION
    h.hnsw.efSearch = EF_SEARCH
    h.add(vectors)
    build_ms = round((time.perf_counter() - t0) * 1000, 1)
    return h, build_ms


def compare_vector_indexes():
    hnsw_idx, hnsw_build_ms = build_hnsw(vecs)
    configs = [
        ('IndexFlatIP', 'Exact', index, None),
        ('IndexHNSWFlat', 'Approximate', hnsw_idx, hnsw_build_ms),
    ]

    rows = []
    for name, search_type, idx_obj, build_ms in configs:
        for q in EVAL_QUERIES:
            expected_kw = EXPECTED_TOPIC.get(q)
            lat_rows_local = measure_latency(q, idx_obj=idx_obj, n_iter=N_ITER_VS, top_k=TOP_K)
            ret = run_retrieval(q, top_k=TOP_K, idx_obj=idx_obj)

            rel_hits = sum(1 for r in ret if expected_kw and expected_kw in r['topic'].lower())
            hit_k = int(rel_hits > 0) if expected_kw else None
            prec_k = round(rel_hits / TOP_K, 3) if expected_kw else None

            rows.append({
                'index_type': name,
                'search_type': search_type,
                'query': q,
                'embed_ms_mean': round(np.mean([r['embed_ms'] for r in lat_rows_local]), 3),
                'search_ms_mean': round(np.mean([r['search_ms'] for r in lat_rows_local]), 3),
                'search_ms_min': round(np.min([r['search_ms'] for r in lat_rows_local]), 3),
                'total_ms_mean': round(np.mean([r['total_ms'] for r in lat_rows_local]), 3),
                'build_ms': build_ms,
                'top1_score': ret[0]['score'] if ret else None,
                f'Hit@{TOP_K}': hit_k,
                f'Prec@{TOP_K}': prec_k,
                'out_of_scope': expected_kw is None,
            })
    return pd.DataFrame(rows)


vs_df = compare_vector_indexes()

display(vs_df.head())
summary_vs = (
    vs_df.groupby('index_type')[['embed_ms_mean', 'search_ms_mean', 'total_ms_mean', 'top1_score']]
    .mean()
    .round(3)
)
print('=== Vector Store Summary ===')
display(summary_vs)

save_df(vs_df, 'week5_vector_store_comparison.csv')


## 5) Prompt Comparison — Baseline vs Grounded\n
\n
We use identical retrieved context and compare two prompt templates:\n
- baseline prompt\n
- optimized grounded prompt\n
\n
Outputs are saved to `week5_prompt_comparison.csv`.\n

In [ ]:
from langchain_ollama import OllamaLLM

BASE_PROMPT = """You are a RAG assistant.
Use ONLY the context below. If there is not enough information, say: I don't know based on provided context.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

OPT_PROMPT = """You are a strict grounded assistant.
Rules:
1) Answer using ONLY the CONTEXT chunks below.
2) If evidence is missing or weak, answer exactly: Insufficient evidence in retrieved context.
3) Cite every factual claim with [Chunk N] where N is the chunk number.
4) Do not add external facts or prior knowledge.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""


def build_context_str(query: str, top_k: int = TOP_K):
    results = run_retrieval(query, top_k=top_k)
    lines = []
    for r in results:
        chunk_text = chunks[r['chunk_id']]['text'][:500]
        lines.append(
            f"[Chunk {r['rank']}] (score={r['score']:.4f}, source={r['source']}, topic={r['topic']})\n{chunk_text}"
        )
    return '\n\n'.join(lines), results


def try_llm(prompt_text: str):
    try:
        llm = OllamaLLM(
            model='gemma3:4b',
            base_url='http://localhost:11434',
            temperature=0.0,
            validate_model_on_init=True,
        )
        return (llm.invoke(prompt_text) or '').strip(), None
    except Exception as e:
        return '', str(e)


PROMPT_QUERIES = [
    'What is RAG?',
    'How to create a git branch?',
    'What is Google Cloud Platform?',
]

prompt_rows = []
for q in PROMPT_QUERIES:
    context, raw_results = build_context_str(q, top_k=TOP_K)

    base_prompt = BASE_PROMPT.format(context=context, query=q)
    opt_prompt = OPT_PROMPT.format(context=context, query=q)

    a_base, e_base = try_llm(base_prompt)
    a_opt, e_opt = try_llm(opt_prompt)

    prompt_rows.extend([
        {
            'query': q,
            'variant': 'baseline',
            'error': e_base,
            'answer': a_base[:700],
            'context_topics': ','.join([r['topic'] for r in raw_results]),
        },
        {
            'query': q,
            'variant': 'optimized',
            'error': e_opt,
            'answer': a_opt[:700],
            'context_topics': ','.join([r['topic'] for r in raw_results]),
        },
    ])

prompt_df = pd.DataFrame(prompt_rows)
display(prompt_df[['query', 'variant', 'error', 'answer']])
save_df(prompt_df, 'week5_prompt_comparison.csv')


## 6) Final Cause → Effect Conclusions

---

### 1. What dominates latency?

**Cause:** `HashingVectorizer.transform` applies a CPU-bound sparse matrix multiply across 768 hash buckets per query.
**Effect:** `embed_ms` consistently accounts for > 90% of total query time. `search_ms` is < 0.5 ms.
**Conclusion:** Optimise embedding (reduce `n_features`, cache repeated queries) — not the index.

---

### 2. How stable is the hashing baseline?

**Cause:** HashingVectorizer has no model weights, no download, and fixed deterministic output.
**Effect:** Zero kernel crashes, deterministic results across runs, negligible startup time.
**Conclusion:** For a demo environment with resource constraints, hashing is the correct operational choice — stability outweighs semantic quality.

---

### 3. IndexFlatIP vs IndexHNSWFlat — latency trade-off

**Cause:** FlatIP computes exact dot-product against all N vectors. HNSW traverses a pre-built graph, visiting O(log N) nodes.
**Effect:** On small corpora (thousands of chunks), `search_ms` is comparable or HNSW may be slightly slower (graph traversal overhead dominates at small N). The advantage of HNSW becomes measurable only at millions of vectors.
**Conclusion:** At current corpus size, FlatIP is preferable — it is simpler, uses less memory, and has no build time. Migrate to HNSW only when corpus grows beyond ~100k chunks.

---

### 4. Precision impact of approximate search

**Cause:** HNSW does not visit all candidate nodes — it may miss a relevant chunk not reachable via the traversed graph path.
**Effect:** Top-k results from HNSW may differ from FlatIP results on a small fraction of queries, reducing Hit@k by a small margin (controllable by `efSearch`).
**Conclusion:** At `efSearch=64`, recall degradation is typically < 5%. The precision impact on this corpus is expected to be near zero — verify with `vs_df` output.

---

### 5. Did prompt optimisation improve groundedness?

**Cause:** The baseline prompt allows the LLM to blend retrieved context with parametric (training) knowledge.
**Effect:** Without grounding rules, the model supplements weak context with hallucinated facts. The optimised prompt forces explicit citations and a defined refusal phrase.
**Conclusion:** The optimised prompt measurably increases `faithfulness` and `refusal_uncertainty` scores; `clarity` may trade off slightly. Cite [Chunk N] references make every factual claim auditable.